<a href="https://www.kaggle.com/code/avikdas567/helios-corn-futures-climate-challenge?scriptVersionId=292044522" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/forecasting-the-future-the-helios-corn-climate-challenge/corn_climate_risk_futures_daily_master.csv
/kaggle/input/forecasting-the-future-the-helios-corn-climate-challenge/corn_regional_market_share.csv


In [2]:
# ============================================================
# Helios Corn Futures Climate Challenge
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.5f}".format)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
DATA_DIR = Path("/kaggle/input/forecasting-the-future-the-helios-corn-climate-challenge")

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
df = pd.read_csv(
    DATA_DIR / "corn_climate_risk_futures_daily_master.csv",
    parse_dates=["date_on"]
)

shares = pd.read_csv(
    DATA_DIR / "corn_regional_market_share.csv"
)

# ------------------------------------------------------------
# Merge regional production weights
# ------------------------------------------------------------
df = df.merge(
    shares[["region_id", "percent_country_production"]],
    on="region_id",
    how="left"
)

df["percent_country_production"] = (
    df["percent_country_production"]
    .fillna(0.0)
    .astype(float) / 100.0
)

# ------------------------------------------------------------
# Climate risk column map
# ------------------------------------------------------------
RISK_MAP = {
    "heat": [
        "climate_risk_cnt_locations_heat_stress_risk_low",
        "climate_risk_cnt_locations_heat_stress_risk_medium",
        "climate_risk_cnt_locations_heat_stress_risk_high",
    ],
    "cold": [
        "climate_risk_cnt_locations_unseasonably_cold_risk_low",
        "climate_risk_cnt_locations_unseasonably_cold_risk_medium",
        "climate_risk_cnt_locations_unseasonably_cold_risk_high",
    ],
    "wet": [
        "climate_risk_cnt_locations_excess_precip_risk_low",
        "climate_risk_cnt_locations_excess_precip_risk_medium",
        "climate_risk_cnt_locations_excess_precip_risk_high",
    ],
    "dry": [
        "climate_risk_cnt_locations_drought_risk_low",
        "climate_risk_cnt_locations_drought_risk_medium",
        "climate_risk_cnt_locations_drought_risk_high",
    ],
}

# ------------------------------------------------------------
# Severity encoding (non-linear)
# ------------------------------------------------------------
for risk, cols in RISK_MAP.items():
    df[f"climate_risk_{risk}_severity_raw"] = (
        0.5 * df[cols[1]] +
        1.0 * df[cols[2]]
    )

# ------------------------------------------------------------
# Production-weighted risk
# ------------------------------------------------------------
for risk in RISK_MAP.keys():
    df[f"climate_risk_{risk}_weighted"] = (
        df[f"climate_risk_{risk}_severity_raw"] *
        df["percent_country_production"]
    )

# ------------------------------------------------------------
# Rolling means (temporal aggregation)
# ------------------------------------------------------------
WINDOWS = [7, 14, 30, 60]

for risk in RISK_MAP.keys():
    for w in WINDOWS:
        df[f"climate_risk_{risk}_ma_{w}d"] = (
            df
            .groupby(["country_name", "region_name"])[f"climate_risk_{risk}_weighted"]
            .transform(lambda x: x.rolling(w, min_periods=3).mean())
        )

# ------------------------------------------------------------
# Momentum features
# ------------------------------------------------------------
for risk in RISK_MAP.keys():
    df[f"climate_risk_{risk}_momentum_7d"] = (
        df
        .groupby(["country_name", "region_name"])[f"climate_risk_{risk}_weighted"]
        .transform(lambda x: x - x.shift(7))
    )

# ------------------------------------------------------------
# Composite stress indices
# ------------------------------------------------------------
df["climate_risk_temperature_stress"] = (
    df["climate_risk_heat_weighted"] +
    df["climate_risk_cold_weighted"]
)

df["climate_risk_moisture_stress"] = (
    df["climate_risk_dry_weighted"] +
    df["climate_risk_wet_weighted"]
)

df["climate_risk_total_stress"] = (
    df["climate_risk_temperature_stress"] +
    df["climate_risk_moisture_stress"]
)

# ------------------------------------------------------------
# Lagged signals (market reaction delay)
# ------------------------------------------------------------
LAGS = [7, 14, 30]

BASE_SIGNALS = [
    "climate_risk_total_stress",
    "climate_risk_temperature_stress",
    "climate_risk_moisture_stress",
]

for col in BASE_SIGNALS:
    for lag in LAGS:
        df[f"{col}_lag_{lag}d"] = (
            df
            .groupby(["country_name", "region_name"])[col]
            .shift(lag)
        )

# ------------------------------------------------------------
# Fill NaNs
# ------------------------------------------------------------
CLIMATE_FEATURES = [c for c in df.columns if c.startswith("climate_risk_")]

df[CLIMATE_FEATURES] = (
    df
    .groupby(["country_name", "region_name"])[CLIMATE_FEATURES]
    .apply(lambda x: x.ffill().bfill())
    .reset_index(drop=True)
)

df[CLIMATE_FEATURES] = df[CLIMATE_FEATURES].fillna(0.0)

# ------------------------------------------------------------
# Build DAILY submission (row-aligned)
# ------------------------------------------------------------
submission = df[
    ["date_on", "country_name", "region_name"] + CLIMATE_FEATURES
].copy()

submission = submission.sort_values(
    ["date_on", "country_name", "region_name"]
).reset_index(drop=True)

submission.insert(0, "ID", submission.index)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
submission.to_csv("submission.csv", index=False)

assert submission.shape[0] == df.shape[0], " Row count mismatch"
assert submission.isna().sum().sum() == 0, " NaNs detected"
assert "ID" in submission.columns, " Missing ID column"

print("submission.csv successfully created")
print("Rows:", submission.shape[0])
print("Columns:", submission.shape[1])
display(submission.head())

submission.csv successfully created
Rows: 320661
Columns: 56


,ID,date_on,country_name,region_name,climate_risk_cnt_locations_heat_stress_risk_low,climate_risk_cnt_locations_heat_stress_risk_medium,climate_risk_cnt_locations_heat_stress_risk_high,climate_risk_cnt_locations_unseasonably_cold_risk_low,climate_risk_cnt_locations_unseasonably_cold_risk_medium,climate_risk_cnt_locations_unseasonably_cold_risk_high,climate_risk_cnt_locations_excess_precip_risk_low,climate_risk_cnt_locations_excess_precip_risk_medium,climate_risk_cnt_locations_excess_precip_risk_high,climate_risk_cnt_locations_drought_risk_low,climate_risk_cnt_locations_drought_risk_medium,climate_risk_cnt_locations_drought_risk_high,climate_risk_heat_severity_raw,climate_risk_cold_severity_raw,climate_risk_wet_severity_raw,climate_risk_dry_severity_raw,climate_risk_heat_weighted,climate_risk_cold_weighted,climate_risk_wet_weighted,climate_risk_dry_weighted,climate_risk_heat_ma_7d,climate_risk_heat_ma_14d,climate_risk_heat_ma_30d,climate_risk_heat_ma_60d,climate_risk_cold_ma_7d,climate_risk_cold_ma_14d,climate_risk_cold_ma_30d,climate_risk_cold_ma_60d,climate_risk_wet_ma_7d,climate_risk_wet_ma_14d,climate_risk_wet_ma_30d,climate_risk_wet_ma_60d,climate_risk_dry_ma_7d,climate_risk_dry_ma_14d,climate_risk_dry_ma_30d,climate_risk_dry_ma_60d,climate_risk_heat_momentum_7d,climate_risk_cold_momentum_7d,climate_risk_wet_momentum_7d,climate_risk_dry_momentum_7d,climate_risk_temperature_stress,climate_risk_moisture_stress,climate_risk_total_stress,climate_risk_total_stress_lag_7d,climate_risk_total_stress_lag_14d,climate_risk_total_stress_lag_30d,climate_risk_temperature_stress_lag_7d,climate_risk_temperature_stress_lag_14d,climate_risk_temperature_stress_lag_30d,climate_risk_moisture_stress_lag_7d,climate_risk_moisture_stress_lag_14d,climate_risk_moisture_stress_lag_30d
0,0,2016-01-01,Brazil,Bahia,1,0,0,1,0,0,1,0,0,1,0,0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.02500,0.01500,0.03033,0.02975,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,-0.03500,0.00000,0.00000,0.00000,0.00000,0.03500,0.00000,0.03500,0.00000,0.00000,0.00000,0.03500,0.00000,0.03500
1,1,2016-01-01,Brazil,Espírito Santo,1,0,0,1,0,0,1,0,0,1,0,0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
2,2,2016-01-01,Brazil,Goiás,24,0,0,24,0,0,24,0,0,16,8,0,0.00000,0.00000,0.00000,4.00000,0.00000,0.00000,0.00000,0.24000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00400,0.00200,0.00000,0.00000,0.00000,0.00700,0.03429,0.01714,0.00800,0.07800,0.00000,0.00000,0.00000,0.24000,0.00000,0.24000,0.24000,0.00000,0.00000,0.03000,0.00000,0.00000,0.00000,0.00000,0.00000,0.03000
3,3,2016-01-01,Brazil,Mato Grosso,29,0,0,29,0,0,29,0,0,12,14,3,0.00000,0.00000,0.00000,10.00000,0.00000,0.00000,0.00000,3.70000,0.00000,0.00000,0.01233,0.00617,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00925,0.52857,0.26429,0.16033,0.90958,0.00000,0.00000,0.00000,3.70000,0.00000,3.70000,3.70000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
4,4,2016-01-01,Brazil,Mato Grosso do Sul,11,0,0,11,0,0,11,0,0,11,0,0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00357,0.00167,0.13417,0.04286,0.10714,0.08167,0.05083,0.00000,0.00000,0.00000,-0.40000,0.00000,0.00000,0.00000,0.40000,0.00000,0.00000,0.00000,0.00000,0.00000,0.40000,0.00000,0.00000
